In [6]:
from datetime import datetime

# Define the query
query = """
SELECT *  
FROM [dbo].[successful_queries]  
WHERE dimension IN ('Completeness', 'Conformity')  
AND FILE_ID > ISNULL((SELECT MAX(FILE_ID) FROM complete_conformity_query_output), 0);
"""

# Execute the query
cursor.execute(query)

# Fetch all results
rows = cursor.fetchall()

# Convert result to DataFrame
columns = [column[0] for column in cursor.description]

# Convert pyodbc.Row objects to list of tuples
rows = [tuple(row) for row in rows]

# Convert result to Spark DataFrame
if rows:
    df_query = spark.createDataFrame(rows, schema=columns)

df_rec =df_query.collect()

In [8]:
# Prepare the list to store dataframes
list_df = []

# Loop through each row in the dataframe
for row in df_rec:
    # Extract values from the row
    file_id = row["file_id"]
    table_name = row["table_name"]
    column_name = row["column_name"]
    rule_no = row["rule_no"]
    business_rule_definition = row["business_rule_definition"].replace("'", '')
    condition = row["query"]
    grp = row["attribute_group"]
    dimension = row["dimension"]
    
    # Create a dictionary for the values
    d = {
        'file_id': file_id, 
        'table_name': table_name,
        'column_name': column_name,
        'rule_no': rule_no,
        'business_rule_definition': business_rule_definition,
        'condition': condition,
        'grp': grp,
        'dimension': dimension
    }

    # Generate a random error_id
    e = str(random.randint(1000, 9999))
    
    # Create the query string
    query = f"""
    SELECT DISTINCT
        {d['file_id']} as file_id,
        '{d['column_name']}' as value,
        {e} AS error_id,
        '{d['rule_no']}' AS rule_no, 
        '{d["business_rule_definition"]}' AS business_rule_definition,
        '{d['table_name']}' AS table_name,
        '{d['column_name']}' AS column_name, 
        concat({d['column_name']}, '') AS column_value,
        concat({d['column_name']}, '') AS business_attribute_name,
        '{d['grp']}' AS attribute_group, 
        'High' AS severity, 
        'YES' AS RequiredAttribute,         
        '{d['dimension']}' AS dimension,
        '{datetime.now()}' as extract_date
    FROM {d['table_name']} 
    WHERE {d['condition']}
    """
    
    # Print the query for debugging purposes
    # print(query)
    try:
        # Execute the query
        cursor.execute(query)
        rows = cursor.fetchall()
        
        # Convert result to DataFrame
        if rows:
            columns = [column[0] for column in cursor.description]
            df_final = pd.DataFrame.from_records(rows, columns=columns)
            list_df.append(df_final)

    except Exception as e:
        print(f"Error executing query: {e}")

In [21]:
error_list = []
# Combine all dataframes into one
if list_df:
    final_df = list_df[0]
    for df in list_df[1:]:
        try:
            final_df = pd.concat([final_df, df], ignore_index=True)
        except:
            error_list.append(df)
    final_df.to_sql('complete_conformity_query_output', con=engine, if_exists='append', index=False)
else:
    print("No dataframes were created from the queries.")